# Job Glue — vendas em PySpark

ETL **raw → curated** rodando no AWS Glue (Spark serverless), como alternativa ao passo
`transformar_para_parquet` da DAG [`11_s3_glue_athena`](../dags/11_s3_glue_athena.py),
que faz a mesma limpeza em pandas dentro do worker do Airflow.

```
s3://vendas-raw/vendas/vendas.csv
        │  DynamicFrame (leitura)
        ▼
  limpeza + colunas derivadas (receita, semana)
        │  Spark DataFrame
        ▼
s3://vendas-curated/vendas/semana=YYYY-MM-DD/*.parquet
        │  enableUpdateCatalog
        ▼
  tabela vendas_db.vendas  →  pronta para o Athena
```

| Em pandas (DAG 11) | Em PySpark (este notebook) |
|---|---|
| Roda num worker só, o CSV inteiro na memória | Roda distribuído; o volume cresce sem reescrever o código |
| Grava um Parquet único | Grava particionado por `semana` |
| Precisa do **Glue Crawler** para registrar a tabela | Registra a tabela sozinho (`enableUpdateCatalog`) |

> **Custo:** sessão interativa e job cobram por DPU/hora, com mínimo de 1 minuto por
> execução. Com 2 workers `G.1X` o gasto é de centavos de dólar, mas **sempre encerre a
> sessão** no fim (`%stop_session`). O `%idle_timeout` abaixo é a rede de segurança para
> quando você fecha o navegador e esquece.

## 1. Configuração da sessão

Estas *magics* só valem no kernel **Glue PySpark** (AWS Glue Studio → *Notebooks*, ou
Jupyter local com `aws-glue-sessions` instalado). Elas precisam vir **antes** de qualquer
código: a sessão nasce na primeira célula Python executada e, depois disso, trocar worker
ou versão exige `%stop_session`.

- `%idle_timeout 30` — derruba a sessão após 30 min parada (evita cobrança esquecida).
- `%glue_version 5.0` — Spark 3.5 / Python 3.11.
- `%worker_type G.1X` + `%number_of_workers 2` — o menor arranjo útil; 2 DPUs.
- `%region` — ajuste para a sua conta. A role usada pela sessão precisa das mesmas
  permissões da DAG 11: S3 nos buckets + `AWSGlueServiceRole`.

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%region us-east-1

## 2. Contexto do Glue

`SparkContext` é o Spark puro; `GlueContext` é a camada da AWS por cima dele, que traz o
`DynamicFrame` (um DataFrame tolerante a schema inconsistente) e os conectores para S3 e
Data Catalog. O objeto `job` controla os *bookmarks* — o mecanismo que faz o Glue
processar só o que chegou desde a última execução.

A função `parametro()` existe porque o mesmo código roda em dois modos: interativo (sem
`sys.argv`, usa os padrões) e como job agendado (recebe `--BUCKET_RAW` e afins).

In [ ]:
import sys

from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql import types as T

contexto_spark = SparkContext.getOrCreate()
contexto_glue = GlueContext(contexto_spark)
spark = contexto_glue.spark_session
job = Job(contexto_glue)


def parametro(nome: str, padrao: str) -> str:
    """Le um --parametro da linha de comando; no modo interativo usa o padrao."""
    if f"--{nome}" in sys.argv:
        return getResolvedOptions(sys.argv, [nome])[nome]
    return padrao


# So existe JOB_NAME quando o notebook roda como job de verdade.
if "--JOB_NAME" in sys.argv:
    argumentos = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job.init(argumentos["JOB_NAME"], argumentos)
    print("modo job:", argumentos["JOB_NAME"])
else:
    print("modo interativo (sem bookmark)")

## 3. Parâmetros

Os mesmos nomes de bucket e banco da stack [`airflow-aws`](../docker-compose.yaml), para
que o notebook e a DAG 11 escrevam no mesmo lugar.

In [ ]:
BUCKET_RAW = parametro("BUCKET_RAW", "vendas-raw")
BUCKET_CURATED = parametro("BUCKET_CURATED", "vendas-curated")
GLUE_DATABASE = parametro("GLUE_DATABASE", "vendas_db")
TABELA_CURATED = parametro("TABELA_CURATED", "vendas")

CAMINHO_RAW = f"s3://{BUCKET_RAW}/vendas/"
CAMINHO_CURATED = f"s3://{BUCKET_CURATED}/vendas/"

print(f"origem:  {CAMINHO_RAW}")
print(f"destino: {CAMINHO_CURATED}")
print(f"tabela:  {GLUE_DATABASE}.{TABELA_CURATED}")

## 4. Leitura da camada raw

O CSV é lido com o schema **declarado**, não inferido. Inferir custa uma passada extra no
arquivo e, pior, muda o tipo de uma coluna conforme o lote do dia — é assim que um job
quebra em produção depois de meses rodando.

As colunas numéricas entram como `string` de propósito: as linhas sujas do `vendas.csv`
(`quantidade` vazia, `preco` zerado) virariam `null` já na leitura e a gente perderia a
chance de contar quantas foram descartadas.

In [ ]:
ESQUEMA_RAW = T.StructType([
    T.StructField("data", T.StringType()),
    T.StructField("produto", T.StringType()),
    T.StructField("categoria", T.StringType()),
    T.StructField("quantidade", T.StringType()),
    T.StructField("preco", T.StringType()),
    T.StructField("regiao", T.StringType()),
])

vendas_raw = (
    spark.read.option("header", "true")
    .schema(ESQUEMA_RAW)
    .csv(CAMINHO_RAW)
)

total_raw = vendas_raw.count()
print(f"{total_raw} linha(s) lidas de {CAMINHO_RAW}")
vendas_raw.show(5, truncate=False)

## 5. Tipagem e colunas derivadas

`cast` devolve `null` quando o valor não converte — é o jeito seguro de tipar em Spark
(o contrário seria o job morrer numa linha ruim). As duas colunas novas:

- **`receita`** = `quantidade * preco`, arredondada em 2 casas;
- **`semana`** = a segunda-feira daquela data. `date_trunc("week", ...)` no Spark já
  trunca para segunda, então dispensa a conta com `weekday` que a DAG 11 faz em pandas.

In [ ]:
vendas_tipada = (
    vendas_raw
    .withColumn("data", F.to_date("data", "yyyy-MM-dd"))
    .withColumn("quantidade", F.col("quantidade").cast(T.IntegerType()))
    .withColumn("preco", F.col("preco").cast(T.DoubleType()))
)

vendas_derivada = (
    vendas_tipada
    .withColumn("receita", F.round(F.col("quantidade") * F.col("preco"), 2))
    .withColumn("semana", F.date_trunc("week", F.col("data")).cast(T.DateType()))
)

vendas_derivada.printSchema()

## 6. Limpeza e contagem do refugo

O filtro é o mesmo da DAG 11: sem `quantidade` ou com `preco` não positivo, a linha sai.
Vale sempre **contar** o que foi descartado em vez de só jogar fora — um salto nesse
número é o primeiro sinal de que a origem mudou.

In [ ]:
condicao_valida = (
    F.col("data").isNotNull()
    & F.col("quantidade").isNotNull()
    & (F.col("quantidade") > 0)
    & F.col("preco").isNotNull()
    & (F.col("preco") > 0)
)

vendas_limpa = vendas_derivada.filter(condicao_valida).cache()

total_limpo = vendas_limpa.count()
descartadas = total_raw - total_limpo
print(f"{total_limpo} linha(s) validas | {descartadas} descartada(s) na limpeza")

# Amostra do que saiu, para conferir que o filtro pegou o que devia.
vendas_derivada.filter(~condicao_valida).show(5, truncate=False)

## 7. Conferência rápida

Uma agregação por categoria só para olhar o resultado antes de gravar — é o mesmo
`SELECT` que a DAG 11 manda para o Athena no fim do pipeline.

In [ ]:
(
    vendas_limpa.groupBy("categoria")
    .agg(
        F.count("*").alias("pedidos"),
        F.round(F.sum("receita"), 2).alias("receita"),
    )
    .orderBy(F.col("receita").desc())
    .show(truncate=False)
)

## 8. Gravação particionada + registro no catálogo

Aqui está a diferença em relação à DAG 11. Em vez de gravar um Parquet e chamar o **Glue
Crawler** para descobrir o schema, o `getSink` com `enableUpdateCatalog=True` cria ou
atualiza a tabela `vendas_db.vendas` na mesma escrita. Um passo a menos e uma cobrança a
menos (o crawler é faturado por minuto de execução).

- `partitionKeys=["semana"]` — o Athena passa a ler só as pastas do período filtrado.
- `updateBehavior="UPDATE_IN_DATABASE"` — mudou a coluna, a tabela acompanha.
- `glueparquet` — o writer Parquet do Glue, que resolve o schema enquanto escreve.

> O banco `vendas_db` precisa existir antes (a DAG 11 o cria, ou
> `aws glue create-database --database-input Name=vendas_db`).

In [ ]:
vendas_dyf = DynamicFrame.fromDF(vendas_limpa, contexto_glue, "vendas_dyf")

destino = contexto_glue.getSink(
    path=CAMINHO_CURATED,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["semana"],
    compression="snappy",
    enableUpdateCatalog=True,
    transformation_ctx="destino_curated",
)
destino.setCatalogInfo(catalogDatabase=GLUE_DATABASE, catalogTableName=TABELA_CURATED)
destino.setFormat("glueparquet")
destino.writeFrame(vendas_dyf)

print(f"gravado: {CAMINHO_CURATED} ({total_limpo} linhas)")

## 9. Validação: ler de volta pelo catálogo

Se a tabela responde pelo Data Catalog, o Athena também enxerga. Ler pelo catálogo (e não
pelo caminho do S3) é o que prova que o registro funcionou.

In [ ]:
conferencia = contexto_glue.create_dynamic_frame.from_catalog(
    database=GLUE_DATABASE,
    table_name=TABELA_CURATED,
    transformation_ctx="conferencia",
).toDF()

print(f"{conferencia.count()} linha(s) na tabela {GLUE_DATABASE}.{TABELA_CURATED}")
conferencia.groupBy("semana").count().orderBy("semana").show(truncate=False)

## 10. Encerramento do job

`job.commit()` grava o *bookmark*: na próxima execução o Glue pula os arquivos já
processados. Sem ele, o job reprocessa tudo sempre.

In [ ]:
vendas_limpa.unpersist()

if "--JOB_NAME" in sys.argv:
    job.commit()
    print("bookmark salvo")
else:
    print("modo interativo: nada a commitar")

## 11. Encerrar a sessão

Rode isto ao terminar. Sem `%stop_session` a sessão fica viva — e cobrando — até bater o
`%idle_timeout` de 30 minutos.

In [ ]:
%stop_session

---

## Transformar em job agendado

No console: **Glue Studio → Notebooks → Save → Actions → Convert to job**. Ou pela CLI,
com o notebook exportado como `.py`:

```bash
aws s3 cp 12_vendas_glue_pyspark.py s3://vendas-curated/scripts/

aws glue create-job \
  --name vendas-etl-pyspark \
  --role AWSGlueServiceRole-vendas \
  --glue-version 5.0 \
  --worker-type G.1X --number-of-workers 2 \
  --command Name=glueetl,PythonVersion=3,ScriptLocation=s3://vendas-curated/scripts/12_vendas_glue_pyspark.py \
  --default-arguments '{"--job-bookmark-option":"job-bookmark-enable","--BUCKET_RAW":"vendas-raw","--BUCKET_CURATED":"vendas-curated","--GLUE_DATABASE":"vendas_db"}'

aws glue start-job-run --job-name vendas-etl-pyspark
```

Para disparar pelo Airflow, troque o `GlueCrawlerOperator` da DAG 11 por:

```python
from airflow.providers.amazon.aws.operators.glue import GlueJobOperator

rodar_glue = GlueJobOperator(
    task_id="rodar_glue",
    job_name="vendas-etl-pyspark",
    aws_conn_id="aws_default",
    wait_for_completion=True,
)
```

## O que não dá para testar de graça

O LocalStack Community emula S3, mas **não** emula Glue nem Athena — o mesmo limite
descrito na DAG 11. Sessões interativas e jobs exigem conta AWS real. Só o bloco de
transformação (células 5 a 7) roda em qualquer Spark local, se você quiser exercitar a
lógica sem gastar nada.